# SAR Matrix — R1 × R2 도킹 스코어 매트릭스

Backbone scaffold에 R1, R2 치환체를 조합하여 모든 R1×R2 쌍의 도킹 스코어를 매트릭스로 생성합니다.

## 결과물
- **SAR Matrix (Heatmap)**: R1 × R2 도킹 스코어
- **Best 조합 자동 식별**
- `sar_matrix.csv` → DataWarrior / Excel
- `sar_matrix.sdf` → DataWarrior (구조 포함)
- `sar_matrix_pymol.pml` → PyMOL 스크립트


## 1. 환경 설정


In [ ]:
#@title 1-1. Install Packages {display-mode: "form"}
!pip install -q rdkit meeko vina openbabel-wheel
!pip install -q pdbfixer openmm pymol-open-source
!pip install -q seaborn pandas matplotlib
print('Packages installed.')


In [ ]:
#@title 1-2. Install smina {display-mode: "form"}
import os, stat, urllib.request, subprocess
BIN_DIR = '/opt/bin' if os.path.exists('/opt/bin/smina') else '/content/bin'
os.makedirs(BIN_DIR, exist_ok=True)

smina_path = os.path.join(BIN_DIR, 'smina')
if not os.path.exists(smina_path):
    print('Downloading smina...')
    urllib.request.urlretrieve(
        'https://sourceforge.net/projects/smina/files/smina.static/download', smina_path)
    os.chmod(smina_path, os.stat(smina_path).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
print(f'smina: {smina_path}')


In [ ]:
#@title 1-3. Imports {display-mode: "form"}
import warnings
warnings.filterwarnings('ignore')

from pymol import cmd
from openbabel import pybel
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, Descriptors
from meeko import MoleculePreparation, PDBQTWriterLegacy
from pdbfixer import PDBFixer
from openmm.app import PDBFile

import os, subprocess, itertools
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

print('All imports OK.')


## 2. Backbone + R-group 정의

### 사용법
1. `CORE_SMILES`: R1, R2 위치를 `[*:1]`, `[*:2]`로 표시
2. `R1_GROUPS`, `R2_GROUPS`: 이름과 SMILES 딕셔너리
3. 자동으로 모든 R1×R2 조합을 생성하여 도킹합니다.


In [ ]:
#@title 2-1. 타겟 단백질 설정 {display-mode: "form"}
TARGET_PDB = "6nzp"    #@param {type:"string"}
TARGET_CHAIN = "A"     #@param {type:"string"}
EXTENDING = 7.0        #@param {type:"number"}
EXHAUSTIVENESS = 32    #@param {type:"integer"}
N_POSES = 5            #@param {type:"integer"}

WORK_DIR = '/content/sar_matrix'
os.makedirs(WORK_DIR, exist_ok=True)
print(f'Target: {TARGET_PDB} chain {TARGET_CHAIN}')


In [ ]:
#@title 2-2. Backbone + R-group 정의 {display-mode: "form"}

# ============================================================
# CORE_SMILES: 치환 위치를 [*:1] (R1), [*:2] (R2)로 표시
# 예시: 4-Anilinoquinazoline 계열
# ============================================================

CORE_SMILES = "c1ccc2ncnc(Nc3cc([*:2])cc([*:1])c3)c2c1"
CORE_NAME = "4-Anilinoquinazoline"

# R1 치환체 (meta 위치)
R1_GROUPS = {
    'H':     '[H]',
    'F':     'F',
    'Cl':    'Cl',
    'CH3':   'C',
    'OCH3':  'OC',
    'CF3':   'C(F)(F)F',
    'NH2':   'N',
}

# R2 치환체 (para 위치)
R2_GROUPS = {
    'H':     '[H]',
    'F':     'F',
    'OH':    'O',
    'NH2':   'N',
    'OCH3':  'OC',
}

n_total = len(R1_GROUPS) * len(R2_GROUPS)
print(f'Core: {CORE_NAME}')
print(f'R1 groups: {len(R1_GROUPS)} → {list(R1_GROUPS.keys())}')
print(f'R2 groups: {len(R2_GROUPS)} → {list(R2_GROUPS.keys())}')
print(f'Total combinations: {n_total}')


In [ ]:
#@title 2-3. R1×R2 전체 조합 SMILES 생성 {display-mode: "form"}

def build_molecule(core_smi, r1_smi, r2_smi, r1_name, r2_name):
    """Core SMILES의 [*:1], [*:2]를 R-group으로 치환하여 완성된 SMILES를 반환."""
    from rdkit.Chem import AllChem, RWMol

    core = Chem.MolFromSmiles(core_smi)
    r1 = Chem.MolFromSmiles(r1_smi)
    r2 = Chem.MolFromSmiles(r2_smi)

    if core is None or r1 is None or r2 is None:
        return None, None

    # Replace dummy atoms with R-groups using ReplaceSubstructs
    # Simpler approach: string replacement then sanitize
    smi = core_smi
    # [*:1] → R1
    if r1_smi == '[H]':
        smi = smi.replace('[*:1]', '[H]')
    else:
        smi = smi.replace('[*:1]', r1_smi)
    # [*:2] → R2
    if r2_smi == '[H]':
        smi = smi.replace('[*:2]', '[H]')
    else:
        smi = smi.replace('[*:2]', r2_smi)

    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None, smi

    # Clean up
    smi_clean = Chem.MolToSmiles(mol)
    return mol, smi_clean


# Generate all combinations
LIGANDS = []
for r1_name, r1_smi in R1_GROUPS.items():
    for r2_name, r2_smi in R2_GROUPS.items():
        mol, smi = build_molecule(CORE_SMILES, r1_smi, r2_smi, r1_name, r2_name)
        if mol is not None:
            name = f'R1={r1_name}_R2={r2_name}'
            LIGANDS.append({
                'name': name, 'R1': r1_name, 'R2': r2_name,
                'smiles': smi, 'mol': mol
            })

print(f'Generated {len(LIGANDS)}/{n_total} valid molecules')

# Show a few
for lig in LIGANDS[:5]:
    print(f"  {lig['name']:25s} {lig['smiles']}")
if len(LIGANDS) > 5:
    print(f'  ... ({len(LIGANDS)-5} more)')


In [ ]:
#@title 2-4. 조합 분자 2D 구조 확인 {display-mode: "form"}
mols = [lig['mol'] for lig in LIGANDS]
legends = [f"R1={lig['R1']}\nR2={lig['R2']}" for lig in LIGANDS]
img = Draw.MolsToGridImage(mols, legends=legends, molsPerRow=len(R2_GROUPS), subImgSize=(250, 200))
img


## 3. 타겟 단백질 준비


In [ ]:
#@title 3-1. PDB 준비 + 도킹 박스 계산 {display-mode: "form"}
os.chdir(WORK_DIR)

# Fetch & clean
cmd.delete('all')
cmd.fetch(TARGET_PDB, type='pdb', path=WORK_DIR)
cmd.remove(f'not chain {TARGET_CHAIN}')
cmd.select('Prot', 'polymer.protein')
cmd.select('Lig', 'organic')
clean_pdb = f'{TARGET_PDB}_clean.pdb'
lig_mol2 = f'{TARGET_PDB}_lig.mol2'
cmd.save(clean_pdb, 'Prot')
cmd.save(lig_mol2, 'Lig')

# Docking box from co-crystal ligand
([minX,minY,minZ],[maxX,maxY,maxZ]) = cmd.get_extent('Lig')
pad = EXTENDING
center = {'x':(maxX+minX)/2, 'y':(maxY+minY)/2, 'z':(maxZ+minZ)/2}
size = {'x':maxX-minX+2*pad, 'y':maxY-minY+2*pad, 'z':maxZ-minZ+2*pad}
cmd.delete('all')

# Fix protein
prot_H = f'{TARGET_PDB}_clean_H.pdb'
fixer = PDBFixer(filename=clean_pdb)
fixer.findMissingResidues()
fixer.findNonstandardResidues()
fixer.replaceNonstandardResidues()
fixer.removeHeterogens(True)
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.addMissingHydrogens(7.4)
with open(prot_H, 'w') as f:
    PDBFile.writeFile(fixer.topology, fixer.positions, f)

# Receptor PDBQT (openbabel + strip ROOT/BRANCH)
rec_qt = prot_H.replace('.pdb', '.pdbqt')
mol = list(pybel.readfile(format='pdb', filename=prot_H))[0]
out = pybel.Outputfile(filename=rec_qt+'.tmp', format='pdbqt', overwrite=True)
out.write(mol); out.close()
skip_tags = ('ROOT','ENDROOT','BRANCH','ENDBRANCH','TORSDOF')
skip_kw = ('torsion','active','between atoms','status')
with open(rec_qt+'.tmp') as f: raw = f.readlines()
with open(rec_qt, 'w') as f:
    for l in raw:
        if l.startswith(skip_tags): continue
        if l.startswith('REMARK') and any(k in l.lower() for k in skip_kw): continue
        f.write(l)
os.remove(rec_qt+'.tmp')

print(f'Receptor: {rec_qt}')
print(f'Box center: ({center["x"]:.1f}, {center["y"]:.1f}, {center["z"]:.1f})')
print(f'Box size:   ({size["x"]:.1f}, {size["y"]:.1f}, {size["z"]:.1f})')


## 4. R1 × R2 전체 조합 도킹


In [ ]:
#@title 4-1. 전체 조합 도킹 {display-mode: "form"}
import time

smina = os.path.join(BIN_DIR, 'smina')
results = []
t0 = time.time()

for i, lig in enumerate(LIGANDS):
    name = lig['name']
    print(f'[{i+1}/{len(LIGANDS)}] {name}...', end=' ', flush=True)

    # 3D embed
    mol = Chem.AddHs(lig['mol'])
    AllChem.EmbedMolecule(mol, randomSeed=42)
    AllChem.MMFFOptimizeMolecule(mol)

    # Ligand PDBQT (meeko)
    lig_qt = os.path.join(WORK_DIR, f'{name}.pdbqt')
    preparator = MoleculePreparation()
    for setup in preparator.prepare(mol):
        pdbqt_str, is_ok, _ = PDBQTWriterLegacy.write_string(setup)
        if is_ok:
            with open(lig_qt, 'w') as f: f.write(pdbqt_str)
            break

    # smina docking
    sdf_out = os.path.join(WORK_DIR, f'{name}_docked.sdf')
    cmd_args = [
        smina, '-r', rec_qt, '-l', lig_qt, '-o', sdf_out,
        '--center_x', str(center['x']), '--center_y', str(center['y']), '--center_z', str(center['z']),
        '--size_x', str(size['x']), '--size_y', str(size['y']), '--size_z', str(size['z']),
        '--exhaustiveness', str(EXHAUSTIVENESS), '--num_modes', str(N_POSES),
        '--cpu', '0',  # use all cores
    ]
    subprocess.run(cmd_args, capture_output=True)

    # Parse best score
    score = None
    if os.path.exists(sdf_out) and os.path.getsize(sdf_out) > 0:
        suppl = list(Chem.SDMolSupplier(sdf_out, removeHs=False))
        if suppl and suppl[0] is not None:
            for prop in suppl[0].GetPropsAsDict():
                if 'affinity' in prop.lower() or 'minimized' in prop.lower():
                    try: score = float(suppl[0].GetProp(prop))
                    except: pass

    ha = mol.GetNumHeavyAtoms()
    mw = Descriptors.MolWt(Chem.RemoveHs(mol))

    results.append({
        'Name': name, 'R1': lig['R1'], 'R2': lig['R2'],
        'SMILES': lig['smiles'],
        'Score': round(score, 2) if score else None,
        'MW': round(mw, 1),
        'cLogP': round(Descriptors.MolLogP(Chem.RemoveHs(mol)), 2),
        'HA': ha,
        'LE': round(-score / ha, 3) if score and ha > 0 else None,
        'SDF': sdf_out,
    })
    if score:
        print(f'{score:.2f} kcal/mol')
    else:
        print('FAILED')

elapsed = time.time() - t0
print(f'\n=== {len([r for r in results if r["Score"]])} / {len(LIGANDS)} docked in {elapsed:.0f}s ===')


## 5. SAR Matrix 생성


In [ ]:
#@title 5-1. SAR Matrix (Pivot Table) {display-mode: "form"}
df = pd.DataFrame(results)

# Pivot: R1 (rows) × R2 (columns) = Score
matrix = df.pivot_table(index='R1', columns='R2', values='Score', aggfunc='first')

# Reorder to match input order
r1_order = [k for k in R1_GROUPS.keys() if k in matrix.index]
r2_order = [k for k in R2_GROUPS.keys() if k in matrix.columns]
matrix = matrix.loc[r1_order, r2_order]

print('=== SAR Matrix (Docking Score, kcal/mol) ===')
print(matrix.to_string(float_format='%.1f'))
print()

# Best combination
best_idx = df.loc[df['Score'].idxmin()]
print(f'Best: R1={best_idx["R1"]}, R2={best_idx["R2"]} → {best_idx["Score"]:.2f} kcal/mol')


In [ ]:
#@title 5-2. SAR Matrix Heatmap {display-mode: "form"}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Score heatmap
sns.heatmap(matrix, annot=True, fmt='.1f', cmap='RdYlGn', center=-7.5,
            linewidths=0.5, ax=axes[0],
            cbar_kws={'label': 'Docking Score (kcal/mol)'})
axes[0].set_title(f'SAR Matrix: {CORE_NAME}\nDocking Score (kcal/mol)')
axes[0].set_ylabel('R1')
axes[0].set_xlabel('R2')

# LE heatmap
le_matrix = df.pivot_table(index='R1', columns='R2', values='LE', aggfunc='first')
le_matrix = le_matrix.loc[r1_order, r2_order]
sns.heatmap(le_matrix, annot=True, fmt='.2f', cmap='YlGn',
            linewidths=0.5, ax=axes[1],
            cbar_kws={'label': 'Ligand Efficiency'})
axes[1].set_title(f'SAR Matrix: {CORE_NAME}\nLigand Efficiency (LE)')
axes[1].set_ylabel('R1')
axes[1].set_xlabel('R2')

plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'sar_matrix_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sar_matrix_heatmap.png')


In [ ]:
#@title 5-3. R1 / R2 개별 효과 분석 {display-mode: "form"}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R1 effect (averaged over R2)
r1_mean = matrix.mean(axis=1).sort_values()
colors = ['#E53935' if v < -8 else '#43A047' if v < -7 else '#FFB300' for v in r1_mean]
axes[0].barh(r1_mean.index, r1_mean.values, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_xlabel('Mean Docking Score (kcal/mol)')
axes[0].set_title('R1 Effect (averaged over R2)')
axes[0].axvline(x=-7.0, color='gray', linestyle='--', alpha=0.5)
for i, (idx, val) in enumerate(r1_mean.items()):
    axes[0].text(val - 0.1, i, f'{val:.1f}', va='center', ha='right', fontsize=9, fontweight='bold', color='white')

# R2 effect (averaged over R1)
r2_mean = matrix.mean(axis=0).sort_values()
colors = ['#E53935' if v < -8 else '#43A047' if v < -7 else '#FFB300' for v in r2_mean]
axes[1].barh(r2_mean.index, r2_mean.values, color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_xlabel('Mean Docking Score (kcal/mol)')
axes[1].set_title('R2 Effect (averaged over R1)')
axes[1].axvline(x=-7.0, color='gray', linestyle='--', alpha=0.5)
for i, (idx, val) in enumerate(r2_mean.items()):
    axes[1].text(val - 0.1, i, f'{val:.1f}', va='center', ha='right', fontsize=9, fontweight='bold', color='white')

plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'sar_r_effects.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
#@title 5-4. R1×R2 상호작용 효과 (Additivity Check) {display-mode: "form"}
# Check if R1 and R2 effects are additive
# If additive: Score(R1,R2) ≈ baseline + effect(R1) + effect(R2)
# Non-additive = synergy or antagonism

baseline = matrix.loc['H', 'H'] if 'H' in matrix.index and 'H' in matrix.columns else matrix.values.mean()
r1_effect = matrix.mean(axis=1) - baseline
r2_effect = matrix.mean(axis=0) - baseline

predicted = pd.DataFrame(
    [[baseline + r1_effect[r1] + r2_effect[r2] for r2 in r2_order] for r1 in r1_order],
    index=r1_order, columns=r2_order
)

residual = matrix - predicted  # positive = worse than predicted, negative = better (synergy)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(residual, annot=True, fmt='.1f', cmap='coolwarm', center=0,
            linewidths=0.5, ax=axes[0],
            cbar_kws={'label': 'Residual (kcal/mol)'})
axes[0].set_title('Additivity Residual\n(negative = synergy, positive = antagonism)')
axes[0].set_ylabel('R1')
axes[0].set_xlabel('R2')

# Scatter: predicted vs actual
axes[1].scatter(predicted.values.flatten(), matrix.values.flatten(), s=60, alpha=0.7)
min_val = min(predicted.values.min(), matrix.values.min()) - 0.5
max_val = max(predicted.values.max(), matrix.values.max()) + 0.5
axes[1].plot([min_val, max_val], [min_val, max_val], 'k--', alpha=0.5, label='Perfect additivity')
axes[1].set_xlabel('Predicted (additive model)')
axes[1].set_ylabel('Actual Score')
axes[1].set_title('Additivity Check')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(WORK_DIR, 'sar_additivity.png'), dpi=150, bbox_inches='tight')
plt.show()

# Report non-additive combinations
print('=== Non-additive combinations (|residual| > 0.5 kcal/mol) ===')
for r1 in r1_order:
    for r2 in r2_order:
        res = residual.loc[r1, r2]
        if abs(res) > 0.5:
            label = 'SYNERGY' if res < 0 else 'ANTAGONISM'
            print(f'  R1={r1}, R2={r2}: {res:+.1f} ({label})')


## 6. 내보내기


In [ ]:
#@title 6-1. CSV 내보내기 (매트릭스 + 전체 테이블) {display-mode: "form"}
# Matrix CSV
matrix_csv = os.path.join(WORK_DIR, 'sar_matrix.csv')
matrix.to_csv(matrix_csv)
print(f'Matrix CSV: {matrix_csv}')

# Full table CSV
table_csv = os.path.join(WORK_DIR, 'sar_table.csv')
df.drop(columns=['SDF'], errors='ignore').to_csv(table_csv, index=False)
print(f'Table CSV:  {table_csv}')

# LE matrix
le_csv = os.path.join(WORK_DIR, 'sar_le_matrix.csv')
le_matrix.to_csv(le_csv)
print(f'LE Matrix:  {le_csv}')

print('\n→ DataWarrior: File > Open > sar_matrix.csv')


In [ ]:
#@title 6-2. SDF 내보내기 (DataWarrior용) {display-mode: "form"}
combined_sdf = os.path.join(WORK_DIR, 'sar_matrix.sdf')
writer = Chem.SDWriter(combined_sdf)

for _, row in df.iterrows():
    sdf_file = row.get('SDF', '')
    if sdf_file and os.path.exists(sdf_file):
        suppl = Chem.SDMolSupplier(sdf_file, removeHs=False)
        if len(suppl) > 0 and suppl[0] is not None:
            mol = suppl[0]
            mol.SetProp('Name', row['Name'])
            mol.SetProp('R1', str(row['R1']))
            mol.SetProp('R2', str(row['R2']))
            mol.SetProp('Score', str(row['Score']))
            mol.SetProp('LE', str(row.get('LE', '')))
            mol.SetProp('MW', str(row['MW']))
            mol.SetProp('cLogP', str(row['cLogP']))
            writer.write(mol)
writer.close()
print(f'Combined SDF: {combined_sdf}')
print('→ DataWarrior: File > Open > sar_matrix.sdf')


In [ ]:
#@title 6-3. PyMOL 스크립트 생성 {display-mode: "form"}
pml = os.path.join(WORK_DIR, 'sar_matrix_pymol.pml')

# Top 5 by score
top5 = df.dropna(subset=['Score']).nsmallest(5, 'Score')

with open(pml, 'w') as f:
    f.write('# SAR Matrix - Top 5 Docking Poses\n')
    f.write('reinitialize\nbg_color white\n\n')
    f.write(f'load {prot_H}, protein\n')
    f.write('show cartoon, protein\ncolor white, protein\n\n')
    colors = ['green', 'cyan', 'yellow', 'salmon', 'orange']
    for i, (_, row) in enumerate(top5.iterrows()):
        sdf = row['SDF']
        name = row['Name'].replace('=', '').replace(' ', '_')
        c = colors[i % len(colors)]
        f.write(f'load {sdf}, {name}\n')
        f.write(f'set all_states, 0\nframe 1\n')
        f.write(f'show sticks, {name}\ncolor {c}, {name}\n')
        f.write(f'# {row["Name"]}: Score={row["Score"]:.2f}, LE={row.get("LE",""):.3f}\n\n')
    f.write(f'select pocket, protein within 5 of {top5.iloc[0]["Name"].replace("=","").replace(" ","_")}\n')
    f.write('show sticks, pocket\ncolor skyblue, pocket\n')
    f.write(f'distance hb, {top5.iloc[0]["Name"].replace("=","").replace(" ","_")}, pocket, mode=2\n')
    f.write('set dash_color, yellow, hb\n')
    f.write(f'zoom {top5.iloc[0]["Name"].replace("=","").replace(" ","_")}, 8\n')

print(f'PyMOL script: {pml}')
print('→ PyMOL: File > Run Script > sar_matrix_pymol.pml')


In [ ]:
#@title 6-4. 결과 다운로드 (Colab) {display-mode: "form"}
import shutil
zip_path = '/content/sar_matrix_results'
shutil.make_archive(zip_path, 'zip', WORK_DIR)
print(f'Archive: {zip_path}.zip')
try:
    from google.colab import files
    files.download(f'{zip_path}.zip')
except ImportError:
    print(f'Not in Colab. Results at: {WORK_DIR}')


## 7. 결과 해석 가이드

### SAR Matrix 읽는 법
- **행(R1)**: 첫 번째 치환 위치의 R-group
- **열(R2)**: 두 번째 치환 위치의 R-group
- **값**: 도킹 스코어 (kcal/mol). 음수가 클수록 강한 결합

### Heatmap 색상
- **초록**: 강한 결합 (좋은 스코어)
- **빨강**: 약한 결합 (나쁜 스코어)

### Additivity Check
- **Residual ≈ 0**: R1과 R2 효과가 독립적 (additive)
- **Residual < -0.5**: 시너지 (R1+R2 조합이 예상보다 좋음)
- **Residual > +0.5**: 길항 (조합이 예상보다 나쁨)

### 다음 단계
1. Best 조합의 도킹 포즈를 PyMOL에서 확인
2. Non-additive 조합의 이유를 구조적으로 분석
3. 새로운 R-group을 추가하여 반복 최적화

